# Publication Experiments: Fair Decision-Focused LearningMain results use **shared HP** (`lr=0.001, steps=200`) for fair comparison.\Appendix results use **per-method best HP** from ablation Exp 5.**Methods**: FPTO, FDFL, SAA, WDRO, PCGrad, MGDA, WS-equal\**Alpha**: [0.5, 2.0] | **Lambda**: [0, 0.5, 1, 5] (static methods) / [0] (MOO methods)\**Fairness**: MAD (paper) + Atkinson (appendix) | **Seeds**: [11, 22, 33]

In [ ]:
# Cell 1: Setup and imports
import os, sys, copy
import pandas as pd
import numpy as np

# --- Path setup (Colab) ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/moo_mtl/colab_upload'
except ImportError:
    DRIVE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

sys.path.insert(0, DRIVE_ROOT)
sys.path.insert(0, os.path.join(DRIVE_ROOT, 'src'))
os.chdir(DRIVE_ROOT)

from configs import ALL_METHOD_CONFIGS, DEFAULT_TRAIN_CFG, make_task_cfg
from fair_dfl.runner import run_experiment_unified
import matplotlib.pyplot as plt

RESULTS_DIR = os.path.join(DRIVE_ROOT, 'results', 'publication')
os.makedirs(RESULTS_DIR, exist_ok=True)

DATA_CSV = os.path.join(DRIVE_ROOT, 'data', 'data_processed.csv')
print(f'DRIVE_ROOT: {DRIVE_ROOT}')
print(f'Results dir: {RESULTS_DIR}')

In [ ]:
# Cell 2: Constants and method configuration

METHODS = ['FPTO', 'FDFL', 'SAA', 'WDRO', 'PCGrad', 'MGDA', 'WS-equal']
ALPHAS = [0.5, 2.0]
LAMBDAS = [0.0, 0.5, 1.0, 5.0]
SEEDS = [11, 22, 33]
N_SAMPLE = 0  # 0 = full dataset (48,784 patients)

# --- Shared HP for main results (fair comparison) ---
SHARED_HP = {
    'lr': 0.001,
    'steps_per_lambda': 200,
}

# --- Per-method best HP from ablation Exp 5 (appendix) ---
BEST_HP = {
    'PCGrad':   {'lr': 0.0005, 'steps_per_lambda': 200},
    'MGDA':     {'lr': 0.005,  'steps_per_lambda': 50},
    'WS-equal': {'lr': 0.001,  'steps_per_lambda': 200},
    'FPTO':     {'lr': 0.001,  'steps_per_lambda': 100},
    'FDFL':     {'lr': 0.001,  'steps_per_lambda': 200},
    'WDRO':     {'lr': 0.001,  'steps_per_lambda': 100},
    'SAA':      {'lr': 0.0001, 'steps_per_lambda': 50},
}


def build_method_configs(methods, hp_overrides=None):
    """Build method_configs dict with optional per-method HP overrides.

    hp_overrides: dict of {method_name: {key: value}} or {key: value} for all.
    """
    configs = {}
    for name in methods:
        cfg = copy.deepcopy(ALL_METHOD_CONFIGS[name])
        cfg['continuation'] = False  # no warm-start across lambda stages
        if hp_overrides is not None:
            if isinstance(hp_overrides, dict) and name in hp_overrides:
                # Per-method overrides
                for k, v in hp_overrides[name].items():
                    cfg[k] = v
            elif isinstance(hp_overrides, dict) and 'lr' in hp_overrides:
                # Shared overrides (flat dict with 'lr', 'steps_per_lambda', etc.)
                for k, v in hp_overrides.items():
                    cfg[k] = v
        configs[name] = cfg
    return configs


def build_experiment_cfg(alpha, fairness_type, lambdas, seeds, n_sample, hp_shared=None):
    """Build experiment config dict for run_experiment_unified()."""
    task_cfg = make_task_cfg(
        data_csv=DATA_CSV,
        n_sample=n_sample,
        alpha_fair=alpha,
        fairness_type=fairness_type,
    )
    train_cfg = copy.deepcopy(DEFAULT_TRAIN_CFG)
    train_cfg['lambdas'] = lambdas
    train_cfg['seeds'] = seeds
    if hp_shared:
        train_cfg.update(hp_shared)
    return {'task': task_cfg, 'training': train_cfg}


def make_summary_table(sdf, group_cols=None):
    """Aggregate test metrics as mean +/- std across seeds.

    Reports normalized regret as the primary 'Regret' column.
    """
    if group_cols is None:
        group_cols = ['config_name']

    has_norm = 'test_regret_normalized' in sdf.columns
    regret_col = 'test_regret_normalized' if has_norm else 'test_regret'

    metrics = [regret_col, 'test_fairness', 'test_pred_mse']
    short = {
        'test_regret_normalized': 'Regret',
        'test_regret': 'Regret',
        'test_fairness': 'Fairness',
        'test_pred_mse': 'Pred MSE',
    }
    rows = []
    for keys, grp in sdf.groupby(group_cols):
        if isinstance(keys, str):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        for m in metrics:
            if m not in grp.columns:
                continue
            mu, sd = grp[m].mean(), grp[m].std()
            row[f'{short[m]}'] = f'{mu:.4f} +/- {sd:.4f}'
            row[f'{short[m]}_mean'] = mu
        row['n_seeds'] = len(grp)
        rows.append(row)
    return pd.DataFrame(rows)


print(f'Methods: {METHODS}')
print(f'Alphas: {ALPHAS}')
print(f'Lambdas: {LAMBDAS}')
print(f'Seeds: {SEEDS}')
print(f'N_SAMPLE: {N_SAMPLE} (0 = full dataset)')

## Main Experiments (Shared HP: lr=0.001, steps=200)Fair comparison — all methods use the same hyperparameters.

In [ ]:
# Cell 4: Run main experiments — MAD fairness
print('=' * 80)
print('MAIN EXPERIMENTS — MAD FAIRNESS (shared HP)')
print('=' * 80)

method_configs_shared = build_method_configs(METHODS, hp_overrides=SHARED_HP)
results_mad = {}

for alpha in ALPHAS:
    print(f'\n--- Alpha = {alpha} ---')
    exp_cfg = build_experiment_cfg(
        alpha=alpha, fairness_type='mad',
        lambdas=LAMBDAS, seeds=SEEDS, n_sample=N_SAMPLE,
        hp_shared=SHARED_HP,
    )
    sdf, idf = run_experiment_unified(exp_cfg, method_configs=method_configs_shared)
    sdf['config_name'] = sdf['method'].map(
        lambda x: {n.lower(): n for n in METHODS}.get(x, x.upper()))
    sdf['alpha_fair'] = alpha
    sdf['fairness_type'] = 'mad'
    sdf['hp_setting'] = 'shared'
    results_mad[alpha] = (sdf, idf)
    print(f'  {len(sdf)} stage rows, {len(idf)} iter rows')

print('\nDone.')

In [ ]:
# Cell 5: Run main experiments — Atkinson fairness (appendix)
print('=' * 80)
print('MAIN EXPERIMENTS — ATKINSON FAIRNESS (shared HP)')
print('=' * 80)

results_atkinson = {}

for alpha in ALPHAS:
    print(f'\n--- Alpha = {alpha} ---')
    exp_cfg = build_experiment_cfg(
        alpha=alpha, fairness_type='atkinson',
        lambdas=LAMBDAS, seeds=SEEDS, n_sample=N_SAMPLE,
        hp_shared=SHARED_HP,
    )
    sdf, idf = run_experiment_unified(exp_cfg, method_configs=method_configs_shared)
    sdf['config_name'] = sdf['method'].map(
        lambda x: {n.lower(): n for n in METHODS}.get(x, x.upper()))
    sdf['alpha_fair'] = alpha
    sdf['fairness_type'] = 'atkinson'
    sdf['hp_setting'] = 'shared'
    results_atkinson[alpha] = (sdf, idf)
    print(f'  {len(sdf)} stage rows, {len(idf)} iter rows')

print('\nDone.')

In [ ]:
# Cell 6: Save main results to CSV

all_stage_mad = pd.concat([sdf for sdf, _ in results_mad.values()], ignore_index=True)
all_stage_atkinson = pd.concat([sdf for sdf, _ in results_atkinson.values()], ignore_index=True)

all_stage_mad.to_csv(os.path.join(RESULTS_DIR, 'stage_main_mad.csv'), index=False)
all_stage_atkinson.to_csv(os.path.join(RESULTS_DIR, 'stage_main_atkinson.csv'), index=False)

# Also save iteration logs
all_iter_mad = pd.concat([idf for _, idf in results_mad.values()], ignore_index=True)
all_iter_atkinson = pd.concat([idf for _, idf in results_atkinson.values()], ignore_index=True)
all_iter_mad.to_csv(os.path.join(RESULTS_DIR, 'iter_main_mad.csv'), index=False)
all_iter_atkinson.to_csv(os.path.join(RESULTS_DIR, 'iter_main_atkinson.csv'), index=False)

print(f'Saved to {RESULTS_DIR}/')
print(f'  stage_main_mad.csv:      {len(all_stage_mad)} rows')
print(f'  stage_main_atkinson.csv: {len(all_stage_atkinson)} rows')

In [ ]:
# Cell 7: Summary tables — Main results (MAD)

for alpha in ALPHAS:
    sdf = results_mad[alpha][0]
    print('=' * 80)
    print(f'  MAD Fairness | alpha={alpha} | Shared HP (lr=0.001, steps=200)')
    print('=' * 80)

    # Per-lambda results
    for lam in sorted(sdf['lambda'].unique()):
        subset = sdf[sdf['lambda'] == lam]
        if len(subset) == 0:
            continue
        tbl = make_summary_table(subset, group_cols=['config_name'])
        tbl = tbl.sort_values('Regret_mean')
        tbl.insert(0, 'Rank', range(1, len(tbl) + 1))
        print(f'\n  lambda = {lam}')
        print(tbl[['Rank', 'config_name', 'Regret', 'Fairness', 'Pred MSE']].to_string(index=False))

    # Overall best per method (across lambdas, by lowest regret)
    print(f'\n  Best lambda per method (lowest regret):')
    for method in METHODS:
        msub = sdf[sdf['config_name'] == method]
        if len(msub) == 0:
            continue
        best_lam_rows = msub.groupby('lambda')['test_regret_normalized' if 'test_regret_normalized' in msub.columns else 'test_regret'].mean()
        best_lam = best_lam_rows.idxmin()
        best_val = best_lam_rows.min()
        print(f'    {method:12s}  lambda={best_lam}  regret={best_val:.4f}')
    print()

In [ ]:
# Cell 8: Summary tables — Atkinson (appendix)

for alpha in ALPHAS:
    sdf = results_atkinson[alpha][0]
    print('=' * 80)
    print(f'  Atkinson Fairness | alpha={alpha} | Shared HP (lr=0.001, steps=200)')
    print('=' * 80)

    for lam in sorted(sdf['lambda'].unique()):
        subset = sdf[sdf['lambda'] == lam]
        if len(subset) == 0:
            continue
        tbl = make_summary_table(subset, group_cols=['config_name'])
        tbl = tbl.sort_values('Regret_mean')
        tbl.insert(0, 'Rank', range(1, len(tbl) + 1))
        print(f'\n  lambda = {lam}')
        print(tbl[['Rank', 'config_name', 'Regret', 'Fairness', 'Pred MSE']].to_string(index=False))
    print()

## Appendix: Per-Method Best HPEach method uses its best LR and steps from ablation Exp 5.| Method | LR | Steps ||--------|------|-------|| PCGrad | 0.0005 | 200 || MGDA | 0.005 | 50 || WS-equal | 0.001 | 200 || FPTO | 0.001 | 100 || FDFL | 0.001 | 200 || WDRO | 0.001 | 100 || SAA | 0.0001 | 50 |

In [ ]:
# Cell 10: Run appendix experiments — MAD with per-method best HP
print('=' * 80)
print('APPENDIX — MAD FAIRNESS (per-method best HP)')
print('=' * 80)

method_configs_tuned = build_method_configs(METHODS, hp_overrides=BEST_HP)
results_tuned_mad = {}

for alpha in ALPHAS:
    print(f'\n--- Alpha = {alpha} ---')
    exp_cfg = build_experiment_cfg(
        alpha=alpha, fairness_type='mad',
        lambdas=LAMBDAS, seeds=SEEDS, n_sample=N_SAMPLE,
    )
    sdf, idf = run_experiment_unified(exp_cfg, method_configs=method_configs_tuned)
    sdf['config_name'] = sdf['method'].map(
        lambda x: {n.lower(): n for n in METHODS}.get(x, x.upper()))
    sdf['alpha_fair'] = alpha
    sdf['fairness_type'] = 'mad'
    sdf['hp_setting'] = 'tuned'
    results_tuned_mad[alpha] = (sdf, idf)
    print(f'  {len(sdf)} stage rows, {len(idf)} iter rows')

print('\nDone.')

In [ ]:
# Cell 11: Run appendix experiments — Atkinson with per-method best HP
print('=' * 80)
print('APPENDIX — ATKINSON FAIRNESS (per-method best HP)')
print('=' * 80)

results_tuned_atkinson = {}

for alpha in ALPHAS:
    print(f'\n--- Alpha = {alpha} ---')
    exp_cfg = build_experiment_cfg(
        alpha=alpha, fairness_type='atkinson',
        lambdas=LAMBDAS, seeds=SEEDS, n_sample=N_SAMPLE,
    )
    sdf, idf = run_experiment_unified(exp_cfg, method_configs=method_configs_tuned)
    sdf['config_name'] = sdf['method'].map(
        lambda x: {n.lower(): n for n in METHODS}.get(x, x.upper()))
    sdf['alpha_fair'] = alpha
    sdf['fairness_type'] = 'atkinson'
    sdf['hp_setting'] = 'tuned'
    results_tuned_atkinson[alpha] = (sdf, idf)
    print(f'  {len(sdf)} stage rows, {len(idf)} iter rows')

print('\nDone.')

In [ ]:
# Cell 12: Save appendix results

all_tuned_mad = pd.concat([sdf for sdf, _ in results_tuned_mad.values()], ignore_index=True)
all_tuned_atkinson = pd.concat([sdf for sdf, _ in results_tuned_atkinson.values()], ignore_index=True)

all_tuned_mad.to_csv(os.path.join(RESULTS_DIR, 'stage_tuned_mad.csv'), index=False)
all_tuned_atkinson.to_csv(os.path.join(RESULTS_DIR, 'stage_tuned_atkinson.csv'), index=False)

# Iteration logs
pd.concat([idf for _, idf in results_tuned_mad.values()], ignore_index=True).to_csv(
    os.path.join(RESULTS_DIR, 'iter_tuned_mad.csv'), index=False)
pd.concat([idf for _, idf in results_tuned_atkinson.values()], ignore_index=True).to_csv(
    os.path.join(RESULTS_DIR, 'iter_tuned_atkinson.csv'), index=False)

print(f'Saved to {RESULTS_DIR}/')
print(f'  stage_tuned_mad.csv:      {len(all_tuned_mad)} rows')
print(f'  stage_tuned_atkinson.csv: {len(all_tuned_atkinson)} rows')

In [ ]:
# Cell 13: Summary tables — Appendix (tuned HP, MAD)

for alpha in ALPHAS:
    sdf = results_tuned_mad[alpha][0]
    print('=' * 80)
    print(f'  MAD Fairness | alpha={alpha} | Per-method best HP')
    print('=' * 80)

    for lam in sorted(sdf['lambda'].unique()):
        subset = sdf[sdf['lambda'] == lam]
        if len(subset) == 0:
            continue
        tbl = make_summary_table(subset, group_cols=['config_name'])
        tbl = tbl.sort_values('Regret_mean')
        tbl.insert(0, 'Rank', range(1, len(tbl) + 1))
        print(f'\n  lambda = {lam}')
        print(tbl[['Rank', 'config_name', 'Regret', 'Fairness', 'Pred MSE']].to_string(index=False))
    print()

## AnalysisPareto front plots, ranking tables, and shared vs tuned HP comparison.

In [ ]:
# Cell 15: Pareto front — Fairness vs Regret (MAD)

fig, axes = plt.subplots(1, len(ALPHAS), figsize=(7 * len(ALPHAS), 6), sharey=False)
if len(ALPHAS) == 1:
    axes = [axes]

COLORS = {
    'FPTO': '#1f77b4', 'FDFL': '#ff7f0e', 'SAA': '#2ca02c', 'WDRO': '#d62728',
    'PCGrad': '#9467bd', 'MGDA': '#8c564b', 'WS-equal': '#e377c2',
}
MARKERS = {
    'FPTO': 'o', 'FDFL': 's', 'SAA': '^', 'WDRO': 'v',
    'PCGrad': 'D', 'MGDA': 'P', 'WS-equal': 'X',
}

for ax, alpha in zip(axes, ALPHAS):
    sdf = results_mad[alpha][0]
    regret_col = 'test_regret_normalized' if 'test_regret_normalized' in sdf.columns else 'test_regret'

    for method in METHODS:
        msub = sdf[sdf['config_name'] == method]
        if len(msub) == 0:
            continue
        # Aggregate per lambda
        agg = msub.groupby('lambda').agg(
            regret_mean=(regret_col, 'mean'),
            regret_std=(regret_col, 'std'),
            fairness_mean=('test_fairness', 'mean'),
            fairness_std=('test_fairness', 'std'),
        ).reset_index()

        color = COLORS.get(method, 'gray')
        marker = MARKERS.get(method, 'o')

        if len(agg) == 1:
            # MOO / baseline method — single point
            ax.scatter(agg['fairness_mean'], agg['regret_mean'],
                      c=color, marker=marker, s=120, zorder=5, label=method,
                      edgecolors='black', linewidths=0.5)
            ax.errorbar(agg['fairness_mean'], agg['regret_mean'],
                       xerr=agg['fairness_std'], yerr=agg['regret_std'],
                       fmt='none', color=color, alpha=0.4, capsize=3)
        else:
            # Static method — Pareto curve
            agg = agg.sort_values('fairness_mean')
            ax.plot(agg['fairness_mean'], agg['regret_mean'],
                   color=color, marker=marker, markersize=8, linewidth=1.5,
                   label=method, zorder=4)
            ax.fill_between(
                agg['fairness_mean'],
                agg['regret_mean'] - agg['regret_std'],
                agg['regret_mean'] + agg['regret_std'],
                color=color, alpha=0.15,
            )
            # Annotate lambda values
            for _, row in agg.iterrows():
                ax.annotate(f'{row["lambda"]:.1f}',
                           (row['fairness_mean'], row['regret_mean']),
                           textcoords='offset points', xytext=(5, 5),
                           fontsize=7, color=color, alpha=0.7)

    ax.set_xlabel('Fairness (MAD, lower = fairer)')
    ax.set_ylabel('Normalized Regret (lower = better)')
    ax.set_title(f'alpha = {alpha}')
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True, alpha=0.3)

fig.suptitle('Pareto Front: Fairness vs Regret (Shared HP)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'pareto_mad.pdf'), bbox_inches='tight', dpi=150)
plt.show()
print(f'Saved pareto_mad.pdf')

In [ ]:
# Cell 16: Ranking consistency across alpha values (MAD, lambda=0)

print('=' * 80)
print('  RANKING CONSISTENCY ACROSS ALPHA (MAD, lambda=0, shared HP)')
print('=' * 80)

regret_col = 'test_regret_normalized'
rank_data = {}

for alpha in ALPHAS:
    sdf = results_mad[alpha][0]
    lam0 = sdf[sdf['lambda'] == 0.0]
    if len(lam0) == 0:
        continue
    means = lam0.groupby('config_name')[regret_col].mean().sort_values()
    rank_data[f'alpha={alpha}'] = {m: r + 1 for r, m in enumerate(means.index)}

if rank_data:
    rank_df = pd.DataFrame(rank_data)
    rank_df.index.name = 'Method'
    rank_df['Avg Rank'] = rank_df.mean(axis=1)
    rank_df = rank_df.sort_values('Avg Rank')
    print(rank_df.to_string())

    print(f'\n--- Regret ordering (best -> worst) ---')
    for col in rank_df.columns[:-1]:
        ranking = rank_df[col].sort_values()
        order = ' > '.join(ranking.index)
        print(f'  {col}: {order}')

In [ ]:
# Cell 17: Shared HP vs Tuned HP comparison (MAD)

print('=' * 100)
print('  SHARED HP vs PER-METHOD BEST HP (MAD, lambda=0)')
print('=' * 100)

regret_col = 'test_regret_normalized'

for alpha in ALPHAS:
    sdf_shared = results_mad[alpha][0]
    sdf_tuned = results_tuned_mad[alpha][0]

    shared_lam0 = sdf_shared[sdf_shared['lambda'] == 0.0]
    tuned_lam0 = sdf_tuned[sdf_tuned['lambda'] == 0.0]

    print(f'\nalpha = {alpha}')
    print(f'  {"Method":12s}  {"Shared":>10s}  {"Tuned":>10s}  {"Delta":>10s}  {"Winner":>8s}')
    print(f'  {"-"*12}  {"-"*10}  {"-"*10}  {"-"*10}  {"-"*8}')

    for method in METHODS:
        s_mean = shared_lam0[shared_lam0['config_name'] == method][regret_col].mean()
        t_mean = tuned_lam0[tuned_lam0['config_name'] == method][regret_col].mean()
        delta = t_mean - s_mean
        winner = 'Tuned' if delta < -0.001 else ('Shared' if delta > 0.001 else 'Tie')
        print(f'  {method:12s}  {s_mean:10.4f}  {t_mean:10.4f}  {delta:+10.4f}  {winner:>8s}')